# Brooklyn ambulance siting with uncertain demand-point locations

Choose five new standby posts from the candidate pool. At a given location, the fastest post responds with probability $1-p$ and the second-fastest distinct post responds with probability $p$. The default $p=0.25$ is an assumption.

Each row of `data/demand_points_all.csv` is a distinct nominal demand point with its own call weight. IDs such as `11201-2` belong to ZIP `11201` and keep their own coordinates and weights. For each point, the notebook tests its nominal coordinates and samples a 500 m ball on a 250 m grid plus four diagonal boundary points. Shifts must stay inside the parent ZIP. For each demand point, take the worst expected drive time among its valid positions, then average those worst times using the point-level call weights. Status quo, greedy, and optimization use this same score.

Run from the project root with a Julia kernel. Use the existing Julia packages plus JSON3 (`import Pkg; Pkg.add("JSON3")`). Set `VALHALLA_MATRIX_URL` to your Valhalla matrix endpoint before routing. It defaults to a local Valhalla service. The first run routes both nominal and shifted locations together. Later runs reuse a cache tied to the coordinates of the sites and demand positions. Set `RADIUS_M = 0.0` for a deterministic run on these demand points.

In [ ]:
# using Pkg
# Pkg.add("JSON3")

    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
   Installed JSON3 ─────── v1.14.4
   Installed StructTypes ─ v1.11.1
    Updating `~/.julia/environments/v1.12/Project.toml`
  [0f8b85d8] + JSON3 v1.14.4
    Updating `~/.julia/environments/v1.12/Manifest.toml`
  [0f8b85d8] + JSON3 v1.14.4
  [856f2bd8] + StructTypes v1.11.1
Precompiling packages...
   1928.1 ms  ✓ StructTypes
   6326.1 ms  ✓ JSON3
  2 dependencies successfully precompiled in 10 seconds. 253 already precompiled.
  99 dependencies precompiled but different versions are currently loaded (ArgTools, Base64, Bzip2_jll, CSV, CodecBzip2, CodecZlib, CommonSubexpressions, Compat, Compat → CompatLinearAlgebraExt, CompilerSupportLibraries_jll, Crayons, DataAPI, DataFrames, DataStructures, DataValueInterfaces, Dates, DiffResults, DiffRules, DocStringExtensions, Downloads, FilePathsBase, FilePathsBase → FilePathsBaseMmapExt, FilePathsBase → FilePathsBaseTestExt, ForwardDiff, Future, Gurobi,

In [3]:
using CSV, DataFrames, JuMP, Gurobi, Printf
using Downloads, JSON3, SHA

## 1. Configuration

In [4]:
DATA_DIR   = "data"
OUTPUT_DIR = "results"

DEMAND_FILE        = joinpath(DATA_DIR, "demand_points_all.csv")        # area_id is the unique demand-point ID
CANDIDATE_FILE     = joinpath(DATA_DIR, "candidate_sites_v2.csv")            # site_id, lat, lon, source (20 pins)
EXISTING_FILE      = joinpath(DATA_DIR, "FDNY_Firehouse_Listing_20261115.csv")  # all NYC firehouses
BOROUGH            = "Brooklyn"

P          = 5               # number of new standby sites to open
WEIGHT_COL = :calls          # w_i: call count per demand point

TIME_LIMIT_SEC = 300
ZIP_FILE = joinpath(DATA_DIR, "modzcta.geojson")
RADIUS_M = 500.0
GRID_SPACING_M = 250.0
BUSY_PROB = 0.25
MATRIX_URL = "https://valhalla1.openstreetmap.de/sources_to_targets"
MAX_SNAP_M = 250
@assert RADIUS_M >= 0 && GRID_SPACING_M > 0 && 0 <= BUSY_PROB <= 1

## 2. Load demand points and sites

In [5]:
isfile(DEMAND_FILE) || error("Missing $DEMAND_FILE. Run from the project root.")
demand = CSV.read(DEMAND_FILE, DataFrame; types=Dict(:area_id => String))
required = [:area_id, :lat, :lon, WEIGHT_COL]
all(c -> c in propertynames(demand), required) ||
    error("Demand points must have area_id, lat, lon, and $(WEIGHT_COL) columns.")
nrow(demand) > 0 || error("No demand points in $DEMAND_FILE.")
any(col -> any(ismissing, demand[!, col]), required) &&
    error("Demand points have missing IDs, coordinates, or call weights.")
demand.area_id = strip.(demand.area_id)
all(id -> occursin(r"^[0-9]{5}(-[0-9]+)?$", id), demand.area_id) ||
    error("Demand-point IDs must be a ZIP or ZIP with a numeric suffix.")
allunique(demand.area_id) || error("Demand-point IDs must be unique.")
all(row -> -90 <= row.lat <= 90 && -180 <= row.lon <= 180, eachrow(demand)) ||
    error("Invalid latitude or longitude in demand points.")

# The suffix identifies a separate point. The prefix is used only for ZIP boundaries.
demand.zip_id = [first(split(id, '-'; limit=2)) for id in demand.area_id]
I_points = demand.area_id
w = Float64.(demand[!, WEIGHT_COL])
all(x -> isfinite(x) && x >= 0, w) && sum(w) > 0 ||
    error("Call weights must be finite and nonnegative with a positive total.")
println("Nominal demand points: ", nrow(demand), " | ZIPs: ", length(unique(demand.zip_id)),
        " | Total calls: ", sum(w))
first(demand, 5)

Nominal demand points: 60 | ZIPs: 37 | Total calls: 202491.0


Row,area_id,lat,lon,calls,zip_id
,SubStrin…,Float64,Float64,Int64,SubStrin…
1,11201,40.6946,-73.9896,5785,11201
2,11201-2,40.6886,-73.991,5675,11201
3,11203,40.6496,-73.9344,4361,11203
4,11203-2,40.6416,-73.9332,2844,11203
5,11204,40.6188,-73.9848,2877,11204


In [6]:
candidates = CSV.read(CANDIDATE_FILE, DataFrame)
candidates.site_id = string.(candidates.site_id)
J_sites = candidates.site_id

# Keep existing firehouses in their input-file order.
firehouses = CSV.read(EXISTING_FILE, DataFrame; normalizenames=true)
firehouses = dropmissing(filter(r -> r.Borough == BOROUGH, firehouses), [:Latitude, :Longitude])
existing = DataFrame(
    site_id = [@sprintf("FH_%03d", k) for k in 1:nrow(firehouses)],
    name    = firehouses.FacilityName,
    address = firehouses.FacilityAddress,
    lat     = Float64.(firehouses.Latitude),
    lon     = Float64.(firehouses.Longitude),
)
K_sites = existing.site_id

isempty(intersect(J_sites, K_sites)) || error("Candidate and existing site_ids overlap")
println("Candidate sites |J| = $(length(J_sites)),  existing firehouses |K| = $(length(K_sites))")
first(candidates, 5)

Candidate sites |J| = 20,  existing firehouses |K| = 66


Row,site_id,lat,lon,zip,area
,String,Float64,Float64,Int64,String
1,24,40.6417,-73.9544,11226,Flatbush
2,25,40.6351,-73.9609,11226,Flatbush
3,26,40.6839,-73.9854,11217,Downtown/Park Slope
4,27,40.6875,-73.9757,11217,Downtown/Park Slope
5,28,40.6883,-73.9325,11221,Bed-Stuy/Bushwick


## 3. Route nominal and shifted demand points

In [7]:
site_ids = vcat(K_sites, J_sites)
n_existing = length(K_sites)
sites = vcat(select(existing, :site_id, :lat, :lon),
             select(candidates, :site_id, :lat, :lon))

Row,site_id,lat,lon
,String,Float64,Float64
1,FH_001,40.6955,-73.9828
2,FH_002,40.6871,-73.9828
3,FH_003,40.7,-73.9923
4,FH_004,40.6931,-73.9971
5,FH_005,40.6721,-73.9997
6,FH_006,40.6804,-74.0064
7,FH_007,40.6518,-74.0053
8,FH_008,40.6746,-73.9767
9,FH_009,40.6718,-73.9872


In [8]:
function in_ring(lon, lat, ring)
    inside = false
    for k in 2:length(ring)
        a, b = ring[k-1][1:2]
        c, d = ring[k][1:2]
        if (b > lat) != (d > lat) && lon < a + (lat - b) * (c - a) / (d - b)
            inside = !inside
        end
    end
    return inside
end

function in_zip(lon, lat, geom)
    polygons = geom.type == "Polygon" ? [geom.coordinates] : geom.coordinates
    return any(poly -> in_ring(lon, lat, first(poly)) &&
                       !any(hole -> in_ring(lon, lat, hole), Iterators.drop(poly, 1)), polygons)
end

function demand_offsets(demand, radius, spacing, zip_file)
    offsets = NamedTuple{(:demand_index, :position_id, :lat, :lon), Tuple{Int, String, Float64, Float64}}[]
    geo = JSON3.read(read(zip_file, String))
    boundaries = Dict(String(f.properties.modzcta) => f.geometry for f in geo.features)
    outside = [row.area_id for row in eachrow(demand)
               if !haskey(boundaries, row.zip_id) ||
                  !in_zip(row.lon, row.lat, boundaries[row.zip_id])]
    isempty(outside) || error("Nominal demand points outside their ZIP or with missing boundaries: $outside")
    radius == 0 && return offsets
    count_steps = floor(Int, radius / spacing)
    shifts = [(a * spacing, b * spacing)
              for a in -count_steps:count_steps for b in -count_steps:count_steps
              if 0 < (a * spacing)^2 + (b * spacing)^2 <= radius^2]
    diagonal = radius / sqrt(2)
    append!(shifts, [(diagonal * a, diagonal * b)
                     for (a, b) in [(-1, -1), (-1, 1), (1, -1), (1, 1)]])
    for (i, row) in enumerate(eachrow(demand)), (k, (dx, dy)) in enumerate(shifts)
        lat = row.lat + dy / 111195
        lon = row.lon + dx / (111195 * cosd(row.lat))
        if in_zip(lon, lat, boundaries[row.zip_id])
            push!(offsets, (demand_index=i, position_id="$(row.area_id)_G$k", lat=lat, lon=lon))
        end
    end
    return offsets
end

matrix_location(point) = (lat=point.lat, lon=point.lon, search_cutoff=MAX_SNAP_M)
function snap_meters(lat, lon, snapped_lat, snapped_lon)
    return 111195 * hypot(lat - snapped_lat, (lon - snapped_lon) * cosd(lat))
end

function route_demand_positions(sites, positions)
    result = fill(NaN, nrow(sites), length(positions))
    isempty(positions) && return result
    # Cache key includes the ordered IDs and actual coordinates of all endpoints.
    site_points = [(id=s.site_id, lat=s.lat, lon=s.lon) for s in eachrow(sites)]
    cache_key = bytes2hex(sha256(JSON3.write((site_points, positions, MATRIX_URL, MAX_SNAP_M))))
    mkpath(OUTPUT_DIR)
    path = joinpath(OUTPUT_DIR, "pr_demand_positions_$(cache_key[1:16]).csv")
    if isfile(path)
        cached = Matrix{Float64}(CSV.read(path, DataFrame))
        size(cached) == size(result) || error("Demand-route cache dimensions do not match")
        return cached
    end
    println("Routing ", length(positions), " demand positions from ", nrow(sites), " sites.")
    for site_batch in Iterators.partition(1:nrow(sites), 2)
        for target_batch in Iterators.partition(eachindex(positions), 48)
            payload = JSON3.write((
                sources=[matrix_location(sites[j, :]) for j in site_batch],
                targets=[matrix_location(positions[k]) for k in target_batch],
                costing="auto",
                costing_options=(auto=(shortest=true, disable_hierarchy_pruning=true),),
                units="kilometers", verbose=true,
            ))
            response_body = IOBuffer()
            response = Downloads.request(MATRIX_URL; method="POST",
                input=IOBuffer(payload), output=response_body, timeout=60,
                headers=["Content-Type" => "application/json",
                         "User-Agent" => "15C57-Brooklyn-EMS-Study/1.0",
                         "X-Client-Id" => "15C57-Brooklyn-EMS-Study"])
            body = String(take!(response_body))
            response.status == 200 || error("Valhalla HTTP $(response.status): $body")
            data = JSON3.read(body)
            data.units == "kilometers" || error("Expected Valhalla distances in kilometers")
            rows = data.sources_to_targets
            length(rows) == length(site_batch) || error("Incomplete Valhalla source batch")
            for (j, routes) in zip(site_batch, rows)
                length(routes) == length(target_batch) || error("Incomplete Valhalla target batch")
                for (k, route) in zip(target_batch, routes)
                    distance = get(route, :distance, nothing)
                    cost = get(route, :cost, nothing)
                    if distance === nothing || !isfinite(distance) || distance < 0
                        continue
                    end
                    cost !== nothing && isfinite(cost) && abs(cost - 1000 * distance) <= 1.1 || continue
                    site, point = sites[j, :], positions[k]
                    any(key -> get(route, key, nothing) === nothing,
                        (:begin_lat, :begin_lon, :end_lat, :end_lon)) && continue
                    snap_meters(site.lat, site.lon, route.begin_lat, route.begin_lon) <= MAX_SNAP_M + 1 || continue
                    snap_meters(point.lat, point.lon, route.end_lat, route.end_lon) <= MAX_SNAP_M + 1 || continue
                    result[j, k] = 4 * distance / 1.609344
                end
            end
        end
        (last(site_batch) % 10 == 0 || last(site_batch) == nrow(sites)) &&
            println("Routed ", last(site_batch), " / ", nrow(sites), " sites.")
    end
    CSV.write(path, DataFrame(result, :auto))
    return result
end


offsets = demand_offsets(demand, RADIUS_M, GRID_SPACING_M, ZIP_FILE)
nominal = [(demand_index=i, position_id=row.area_id, lat=Float64(row.lat), lon=Float64(row.lon))
           for (i, row) in enumerate(eachrow(demand))]
positions = vcat(nominal, offsets)
route_times = route_demand_positions(sites, positions)

# Require complete routing to every nominal point. Use only fully routed shifts
# so all layouts face the same uncertainty set.
valid = [all(isfinite, view(route_times, :, k)) for k in eachindex(positions)]
invalid_nominal = [nominal[k].position_id for k in eachindex(nominal) if !valid[k]]
isempty(invalid_nominal) ||
    error("Incomplete routes to nominal demand points: $invalid_nominal. Check these endpoints before solving.")
dropped_shifts = [positions[k].position_id for k in length(nominal)+1:length(positions) if !valid[k]]
isempty(dropped_shifts) || println("Shifts excluded by routing checks: ", join(dropped_shifts, ", "))
all_times = route_times[:, valid]
valid_positions = positions[valid]
point_ids = [p.position_id for p in valid_positions]
point_demand = [p.demand_index for p in valid_positions]
points_by_demand = [findall(==(i), point_demand) for i in eachindex(I_points)]
println("Nominal points: ", length(I_points), " | Routed shifts: ", count(valid) - length(I_points),
        " | Nominal points with shifts: ", count(s -> length(s) > 1, points_by_demand))

Routing 953 demand positions from 86 sites.
Routed 10 / 86 sites.
Routed 20 / 86 sites.
Routed 30 / 86 sites.
Routed 40 / 86 sites.
Routed 50 / 86 sites.
Routed 60 / 86 sites.
Routed 70 / 86 sites.
Routed 80 / 86 sites.
Routed 86 / 86 sites.
Shifts excluded by routing checks: 11201_G7, 11201_G8, 11233_G13
Nominal points: 60 | Routed shifts: 890 | Nominal points with shifts: 60


## 4. Choose five sites

For each nominal demand point, `worst[i]` bounds the expected drive time at each of its possible positions. Primary and backup assignments may change with the position. Open-post decisions are shared. For each position, only the fastest two existing posts and candidates at least as fast as the second existing post need to be considered.

In [9]:
function solve_standby_model(times, point_demand, weights, n_existing, P, p)
    n_sites, n_points = size(times)
    model = Model(Gurobi.Optimizer)
    set_silent(model)
    set_time_limit_sec(model, TIME_LIMIT_SEC)
    @variable(model, y[1:n_sites], Bin)
    @variable(model, worst[1:length(weights)] >= 0)
    @constraint(model, [j in 1:n_existing], y[j] == 1)
    @constraint(model, sum(y[j] for j in n_existing+1:n_sites) == P)

    for k in 1:n_points
        existing_two = partialsortperm(view(times, 1:n_existing, k), 1:2)
        cutoff = times[existing_two[2], k]
        eligible = vcat(existing_two,
            [j for j in n_existing+1:n_sites if times[j, k] <= cutoff])
        primary = @variable(model, [eligible], Bin)
        backup = @variable(model, [eligible], Bin)
        @constraint(model, sum(primary[j] for j in eligible) == 1)
        @constraint(model, sum(backup[j] for j in eligible) == 1)
        @constraint(model, [j in eligible], primary[j] + backup[j] <= y[j])
        first_time = sum(times[j, k] * primary[j] for j in eligible)
        second_time = sum(times[j, k] * backup[j] for j in eligible)
        @constraint(model, first_time <= second_time)
        @constraint(model, worst[point_demand[k]] >= (1-p) * first_time + p * second_time)
    end

    @objective(model, Min, sum(weights[i] * worst[i] for i in eachindex(weights)) / sum(weights))
    optimize!(model)
    is_solved_and_feasible(model) || error("Solve failed: $(termination_status(model))")
    selected = [j for j in n_existing+1:n_sites if value(y[j]) > 0.5]
    return (selected=selected, objective=objective_value(model))
end

sol = solve_standby_model(all_times, point_demand, w, n_existing, P, BUSY_PROB)
println("Selected candidate pins: ", site_ids[sol.selected])
@printf("Worst-case expected drive time: %.6f minutes per call\n", sol.objective)

Set parameter Username
Set parameter LicenseID to value 2857454
Academic license - for non-commercial use only - expires 2027-08-26
Selected candidate pins: ["26", "35", "39", "40", "41"]
Worst-case expected drive time: 3.824769 minutes per call


## 5. Selected sites and worst-case positions by demand point

In [10]:
function score_layout(selected)
    open_sites = vcat(collect(1:n_existing), selected)
    ranked = [open_sites[partialsortperm(view(all_times, open_sites, k), 1:2)]
              for k in axes(all_times, 2)]
    expected = [(1-BUSY_PROB) * all_times[pair[1], k] + BUSY_PROB * all_times[pair[2], k]
                for (k, pair) in enumerate(ranked)]
    worst_points = [points[argmax(expected[points])] for points in points_by_demand]
    return (
        worst=sum(w .* expected[worst_points]) / sum(w),
        ranked=ranked, expected=expected, worst_points=worst_points,
    )
end

best = score_layout(sol.selected)
existing_only = score_layout(Int[])
@assert isapprox(best.worst, sol.objective; atol=1e-6)
@printf("Existing-only worst-case score: %.6f min\n", existing_only.worst)
@printf("Worst-case savings vs. existing posts: %.3f seconds per call\n",
        60 * (existing_only.worst - best.worst))

selected_sites = candidates[sol.selected .- n_existing, :]
display(selected_sites)
assignment = DataFrame(
    area_id=I_points,
    zip_id=demand.zip_id,
    calls=w,
    worst_position=point_ids[best.worst_points],
    worst_primary=[site_ids[best.ranked[k][1]] for k in best.worst_points],
    worst_backup=[site_ids[best.ranked[k][2]] for k in best.worst_points],
    worst_expected_minutes=best.expected[best.worst_points],
)
mkpath(OUTPUT_DIR)
CSV.write(joinpath(OUTPUT_DIR, "selected_sites_pr.csv"), selected_sites)
CSV.write(joinpath(OUTPUT_DIR, "area_assignment_pr.csv"), assignment)

Existing-only worst-case score: 3.988847 min
Worst-case savings vs. existing posts: 9.845 seconds per call


Row,site_id,lat,lon,zip,area
,String,Float64,Float64,Int64,String
1,26,40.6839,-73.9854,11217,Downtown/Park Slope
2,35,40.6712,-73.9428,11213,Between Flatbush and Bed-Stuy/Bushwick
3,39,40.6365,-74.0143,11220,Bay Ridge/Sunset Park
4,40,40.6376,-73.9069,11236,Canarsie/Flatlands
5,41,40.578,-73.9829,11224,Coney Island/Brighton


"results/area_assignment_pr.csv"

In [11]:
function greedy_layout(P)
    selected = Int[]
    remaining = collect((n_existing + 1):length(site_ids))

    for _ in 1:P
        scores = [score_layout(vcat(selected, j)).worst for j in remaining]
        best_index = argmin(scores)
        push!(selected, splice!(remaining, best_index))
    end

    return selected
end

layouts = [Int[], greedy_layout(P), sol.selected]
worst_minutes = [score_layout(layout).worst for layout in layouts]

comparison = DataFrame(
    method = ["Status quo", "Greedy", "Optimization"],
    selected_pins = [join(site_ids[layout], ", ") for layout in layouts],
    worst_case_minutes = round.(worst_minutes; digits=4),
    seconds_saved_vs_status_quo =
        round.(60 .* (worst_minutes[1] .- worst_minutes); digits=3),
)

display(comparison)

@printf("Optimization saves %.3f seconds per call vs. greedy\n",
        60 * (worst_minutes[2] - worst_minutes[3]))

CSV.write(joinpath(OUTPUT_DIR, "comparison_pr.csv"), comparison)

Row,method,selected_pins,worst_case_minutes,seconds_saved_vs_status_quo
,String,String,Float64,Float64
1,Status quo,,3.9888,0.0
2,Greedy,"35, 41, 40, 39, 26",3.8248,9.845
3,Optimization,"26, 35, 39, 40, 41",3.8248,9.845


Optimization saves 0.000 seconds per call vs. greedy


"results/comparison_pr.csv"